# Import

In [2]:
import os
import re
import time
import json
import math
import random
import pickle
import numpy as np
import pandas as pd
import scanpy as sc
import tifffile as tiff

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset,DataLoader
from pytorch_metric_learning.losses import ArcFaceLoss,SubCenterArcFaceLoss

from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score,f1_score

import warnings
warnings.filterwarnings("ignore",category=FutureWarning)

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Model components

In [2]:
class BasicBlock(nn.Module):
    expansion=1
    def __init__(self,in_channel,out_channel,stride=1,downsample=None,**kwargs):
        super(BasicBlock,self).__init__()
        self.conv1=nn.Conv2d(in_channels=in_channel,out_channels=out_channel,
                               kernel_size=3,stride=stride,padding=1,bias=False)
        self.bn1=nn.BatchNorm2d(out_channel)
        self.relu=nn.ReLU()
        self.conv2=nn.Conv2d(in_channels=out_channel,out_channels=out_channel,
                               kernel_size=3,stride=1,padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(out_channel)
        self.downsample=downsample

    def forward(self,x):
        identity=x
        if self.downsample is not None:
            identity=self.downsample(x)

        out=self.conv1(x)
        out=self.bn1(out)
        out=self.relu(out)

        out=self.conv2(out)
        out=self.bn2(out)

        out+=identity
        out=self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self,block,blocks_num,data_channel,
                 num_classes=1000,include_top=True,groups=1,width_per_group=64):
        super(ResNet,self).__init__()
        self.data_channel=data_channel
        self.include_top=include_top
        self.in_channel=64

        self.groups=groups
        self.width_per_group=width_per_group

        self.conv1=nn.Conv2d(data_channel,self.in_channel,kernel_size=7,stride=2,padding=3,bias=False)
        self.bn1=nn.BatchNorm2d(self.in_channel)
        self.relu=nn.ReLU(inplace=True)
        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2,padding=1)
        self.layer1=self._make_layer(block,64,blocks_num[0])
        self.layer2=self._make_layer(block,128,blocks_num[1],stride=2)
        self.layer3=self._make_layer(block,256,blocks_num[2],stride=2)
        self.layer4=self._make_layer(block,512,blocks_num[3],stride=2)
        if self.include_top:
            self.avgpool=nn.AdaptiveAvgPool2d((1,1))  # output size=(1,1)
            self.fc=nn.Linear(512 * block.expansion,num_classes)

        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')

    def _make_layer(self,block,channel,block_num,stride=1):
        downsample=None
        if stride!=1 or self.in_channel!=channel * block.expansion:
            downsample=nn.Sequential(
                nn.Conv2d(self.in_channel,channel * block.expansion,kernel_size=1,stride=stride,bias=False),
                nn.BatchNorm2d(channel * block.expansion))

        layers=[]
        layers.append(block(self.in_channel,channel,
                            downsample=downsample,stride=stride,
                            groups=self.groups,width_per_group=self.width_per_group))
        self.in_channel=channel * block.expansion

        for _ in range(1,block_num):
            layers.append(block(self.in_channel,channel,
                                groups=self.groups,width_per_group=self.width_per_group))
        return nn.Sequential(*layers)

    def forward(self,x):
        x=self.conv1(x)
        x=self.bn1(x)
        x=self.relu(x)
        x=self.maxpool(x)

        x=self.layer1(x)
        x=self.layer2(x)
        x=self.layer3(x)
        x=self.layer4(x)

        if self.include_top:
            x=self.avgpool(x)
            x=torch.flatten(x, 1)
            x=self.fc(x)
        return x

def resnet34(data_channel,num_classes=1000,include_top=True):
    return ResNet(BasicBlock,[3,4,6,3],data_channel,num_classes=num_classes,include_top=include_top)

In [3]:
class CrossAttention(nn.Module):
    def __init__(self,feat_dim,pos_dim):
        super().__init__()
        self.query=nn.Conv2d(feat_dim,feat_dim,1)
        self.key=nn.Conv2d(pos_dim,feat_dim,1)
        self.value=nn.Conv2d(pos_dim,feat_dim,1)

        self.alpha=nn.Parameter(torch.ones(1))
    def forward(self,feat,pos):
        Q=self.query(feat)  #[B,C,H,W]
        K=self.key(pos)     #[B,C,H,W]
        V=self.value(pos)   #[B,C,H,W]

        attn=torch.einsum('bchw,bchw->bhw',Q,K)  #[B,H,W]
        attn=torch.sigmoid(attn).unsqueeze(1)  #[B,1,H,W]
        return feat+self.alpha*(attn*V)

# Model

In [4]:
class DualStreamPositionNet(nn.Module):
    def __init__(self,feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate=0):
        super().__init__()
        self.feat_conv=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,7,padding=3),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU()
        )
        self.shortcut=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,1),
            nn.BatchNorm2d(hid_dim)
        )

        self.cross_attn=CrossAttention(hid_dim,pos_dim)
        self.ResNet=resnet34(hid_dim)        
        self.ResNet.fc=nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(self.ResNet.fc.in_features,emb_dim,bias=False)
        )

    def forward(self,feat,pos):
        residual=self.shortcut(feat)
        feat=self.feat_conv(feat)
        feat=feat+residual

        fused=self.cross_attn(feat,pos)
        embedding=self.ResNet(fused)
        embedding=nn.functional.normalize(embedding,p=2,dim=-1)
        return embedding

# TiffDataset

In [5]:
class TiffDataset(Dataset):
    def __init__(self,root_dir,background_dir,embedding_type,transform=None,classes=None):
        self.root_dir=root_dir
        self.background_dir=background_dir
        self.transform=transform
        self.samples=[]
        if classes is not None:
            self.classes=classes
        else:
            self.classes=self._find_classes(self.root_dir)

        for class_name in self.classes:
            class_dir=os.path.join(root_dir,str(class_name))
            if os.path.isdir(class_dir):
                for fname in os.listdir(class_dir):
                    if fname.endswith(".tiff") or fname.endswith(".tif"):
                        self.samples.append((os.path.join(class_dir,fname),class_name))

        self.background_emb=tiff.imread(os.path.join(background_dir,f'{embedding_type}_background_emb.tiff')).astype('float32')
        self.background_emb=torch.tensor(self.background_emb)

        self.background_pos=tiff.imread(os.path.join(background_dir,f'{embedding_type}_background_pos.tiff')).astype('float32')
        self.background_pos=torch.tensor(self.background_pos)
        x,y=self.background_pos[0],self.background_pos[1]
        self.min_x=torch.min(x)
        self.max_x=torch.max(x)
        self.min_y=torch.min(y)
        self.max_y=torch.max(y)

    def _find_classes(self,root_dir):
        classes=sorted([int(d.name) for d in os.scandir(root_dir) if d.is_dir()])
        return classes

    def __len__(self):
        return len(self.samples)

    def _pos_encoding(self,pos):
        x,y=pos[0],pos[1]
        mask=(x==0)&(y==0)
        x=(x-self.min_x)/(self.max_x-self.min_x)
        y=(y-self.min_y)/(self.max_y-self.min_y)

        num_freq=8
        div_term=torch.exp(torch.arange(0,num_freq)*(-np.log(10000.0)/num_freq))
        x_enc=torch.cat([torch.sin(x.unsqueeze(-1)*div_term),torch.cos(x.unsqueeze(-1)*div_term)],dim=-1).permute(2,0,1)
        y_enc=torch.cat([torch.sin(y.unsqueeze(-1)*div_term),torch.cos(y.unsqueeze(-1)*div_term)],dim=-1).permute(2,0,1)

        combined_enc=torch.cat([x_enc,y_enc],dim=0)
        combined_enc=combined_enc.masked_fill(mask.unsqueeze(0),0)
        return combined_enc

    def __getitem__(self,idx):
        file_path,label=self.samples[idx]
        image=tiff.imread(file_path).astype('float32')
        image=torch.tensor(image)

        comb_channels=torch.cat((self.background_pos,self.background_emb,image),dim=0)
        if self.transform:
            comb_channels=self.transform(comb_channels)

        feature_channels=comb_channels[self.background_pos.shape[0]:]
        pos_channels=comb_channels[:self.background_pos.shape[0]]

        pos_channels=self._pos_encoding(pos_channels)
        return {'feature_channels':feature_channels,'pos_channels':pos_channels,'label':torch.tensor(label)}

In [6]:
def set_interpolation(transform_list,mode):
    new_transforms=[]
    for t in transform_list:
        if hasattr(t,'interpolation'):
            t.interpolation=mode
        new_transforms.append(t)
    return transforms.Compose(new_transforms)

data_transform={
    'train': transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.3,1.0)),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.RandomVerticalFlip(), # theoretically also justified
                                 ##transforms.RandomRotation(30), # would cause much slower convergence
                                ]),
    'val': transforms.Compose([transforms.Resize(256),
                                transforms.CenterCrop(224),
                              ]),
    'test': transforms.Compose([transforms.Resize(256),
                                transforms.CenterCrop(224),])}

for trans_name,trans in data_transform.items():
    data_transform[trans_name]=set_interpolation(trans.transforms,InterpolationMode.BILINEAR)

# For Train & Test

In [7]:
def test(model,test_loaders,loss_function,device):
    to_return_metrics={'accuracy':{},'f1_score':{}}
    model.eval()
    loss_function.eval()
    with torch.no_grad():
        for species,test_loader in test_loaders:
            all_labels,all_preds,all_probs=[],[],[]
            for data in tqdm(test_loader,desc=f"Testing {species}",leave=False):
                data={k:v.to(torch.float32).to(device) for k, v in data.items()}
                labels=data['label'].to(torch.int64)

                embeddings=model(data['feature_channels'],data['pos_channels'])
                logits=loss_function[species].get_cosine(embeddings)*loss_function[species].scale
                probs=torch.softmax(logits,dim=1)
                _,preds=torch.max(probs,1)

                all_labels.append(labels.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
            all_labels=np.concatenate(all_labels)
            all_preds=np.concatenate(all_preds)
            all_probs=np.concatenate(all_probs)

            accuracy=accuracy_score(all_labels,all_preds)
            f1=f1_score(all_labels,all_preds,average='weighted')

            to_return_metrics['accuracy'][species]=round(accuracy,4)
            to_return_metrics['f1_score'][species]=round(f1,4)

            print(f'Species: {species}')
            print('accuracy:{:.4f}, f1_score:{:.4f}'.format(accuracy,f1))
        torch.cuda.empty_cache()
        return to_return_metrics

In [8]:
def train(model,train_loaders,val_loaders,loss_function,optimizer,device,epochs):
    metrics_by_epoch={}
    torch.cuda.empty_cache()
    for epoch in range(epochs):
        total_loss=0
        model.train()
        loss_function.train()

        species_list=[species for species,dl in train_loaders]
        iterators=[iter(dl) for species,dl in train_loaders]
        pseudo_combined_batch_steps=len(species_list)

        total_steps=sum([len(dl) for species,dl in train_loaders])
        progress_bar=tqdm(total=total_steps,desc=f"Epoch {epoch+1}")
        i=0
        while True:
            selected_idx=i% pseudo_combined_batch_steps
            try:
                data=next(iterators[selected_idx])
            except StopIteration:
                break

            species=species_list[selected_idx]
            data={k:v.to(torch.float32).to(device) for k,v in data.items()}
            labels=data['label'].to(torch.int64)
            embeddings=model(data['feature_channels'],data['pos_channels'])
            loss=loss_function[species](embeddings,labels)/pseudo_combined_batch_steps
            loss.backward()

            i+=1
            if i%pseudo_combined_batch_steps==0:
                optimizer.step()
                optimizer.zero_grad()
            total_loss+=loss.item()*pseudo_combined_batch_steps
            progress_bar.update(1)
            progress_bar.set_postfix(loss=total_loss/progress_bar.n)
        progress_bar.close()
        print('epoch {}, loss:{:.4f}'.format(epoch+1,total_loss/total_steps))

        if epoch%5==4:
            print('At epoch '+str(epoch+1),':')
            metrics_by_epoch[epoch+1]=test(model,val_loaders,loss_function,device)
            torch.save(model.state_dict(),'./model/model_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')
            torch.save(loss_function.state_dict(),'./model/loss_func_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')
    with open('./model/metrics_by_epoch_'+str(int(time.time()))+'.json','w') as f:
        json.dump(metrics_by_epoch,f,indent=4)

# Train & Test

## frog & zebrafish

### dataloader

In [9]:
#frog_zebrafish dataset building
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
all_species=['frog','zebrafish']

train_set,val_set,test_set={},{},{}

restrict_classes=None
background_image_path=f'./use_data/Heatmap_Paintings/{mission_name}'
for species in all_species:
    for _set,set_type in [(train_set,'train'),(val_set,'val'),(test_set,'test')]:
        image_path=f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_{species}/{set_type}'
        _set[species]=TiffDataset(root_dir=image_path,background_dir=background_image_path,embedding_type=embedding_type,transform=data_transform[set_type],classes=restrict_classes)
        if set_type=='train':
            restrict_classes=_set[species].classes
        elif set_type=='test':
            restrict_classes=None

classes_of_species={}
for species in all_species:
    classes_of_species[species]=train_set[species].classes

In [10]:
#frog_zebrafish dataloader building
all_species=['frog','zebrafish']

train_loaders,val_loaders,test_loaders=[],[],[]
for species in all_species:
    for _loaders,_set,_type in [(train_loaders,train_set,'train'),(val_loaders,val_set,'val'),(test_loaders,test_set,'test')]:
        _loader=DataLoader(dataset=_set[species],batch_size=32,shuffle=True,drop_last=(_type=='train'))
        _loaders.append((species,_loader))

### train

In [11]:
feat_dim=train_set[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
dropout_rate=0 #dropout in metric learning is much less recommended than in supervised learning
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate).to(device)

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=100
train(model,train_loaders,val_loaders,loss_function,optimizer,device,epochs)

Epoch 1: 100%|██████████████████████████████████████████████████████████| 2250/2250 [38:42<00:00,  1.03s/it, loss=8.95]


epoch 1, loss:8.9524


Epoch 2: 100%|██████████████████████████████████████████████████████████| 2250/2250 [40:19<00:00,  1.08s/it, loss=8.26]


epoch 2, loss:8.2562


Epoch 3: 100%|██████████████████████████████████████████████████████████| 2250/2250 [40:20<00:00,  1.08s/it, loss=7.51]


...

epoch 99, loss:1.1967


Epoch 100: 100%|████████████████████████████████████████████████████████| 2250/2250 [43:23<00:00,  1.16s/it, loss=1.17]


epoch 100, loss:1.1675
At epoch 100 :


Species: frog
accuracy:0.7678, f1_score:0.7631


Species: zebrafish
accuracy:0.8438, f1_score:0.8424


### test

In [12]:
#test 95 epoch model
feat_dim=train_set[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/frog_zebrafish_95_epoch_model.pkl",weights_only=True))

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/frog_zebrafish_95_epoch_loss_func.pkl",weights_only=True))

_=test(model,test_loaders,loss_function,device)

Species: frog
accuracy:0.7655, f1_score:0.7618


Species: zebrafish
accuracy:0.8604, f1_score:0.8577


## human & mouse & lemur

### dataloader

In [13]:
mission_name='Human_Mouse_Lemur_2000hv_15000cell'
embedding_type='ESM1b'
all_species=['human','mouse','lemur']

train_set2,val_set2,test_set2={},{},{}

restrict_classes=None
background_image_path=f'./use_data/Heatmap_Paintings/{mission_name}'
for species in all_species:
    for _set,set_type in [(train_set2,'train'),(val_set2,'val'),(test_set2,'test')]:
        image_path=f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_{species}/{set_type}'
        _set[species]=TiffDataset(root_dir=image_path,background_dir=background_image_path,embedding_type=embedding_type,transform=data_transform[set_type],classes=restrict_classes)
        if set_type=='train':
            restrict_classes=_set[species].classes
        elif set_type=='test':
            restrict_classes=None

classes_of_species={}
for species in all_species:
    classes_of_species[species]=train_set2[species].classes

In [14]:
all_species=['human','mouse','lemur']

train_loaders2,val_loaders2,test_loaders2=[],[],[]
for species in all_species:
    for _loaders,_set,_type in [(train_loaders2,train_set2,'train'),(val_loaders2,val_set2,'val'),(test_loaders2,test_set2,'test')]:
        _loader=DataLoader(dataset=_set[species],batch_size=32,shuffle=True,drop_last=(_type=='train'))
        _loaders.append((species,_loader))

### train

In [14]:
feat_dim=train_set2[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set2[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
dropout_rate=0 #dropout in metric learning is much less recommended than in supervised learning
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate).to(device)

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'human':3,'mouse':3,'lemur':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=100
train(model,train_loaders2,val_loaders2,loss_function,optimizer,device,epochs)

Epoch 1: 100%|████████████████████████████████████████████████████████████| 843/843 [15:13<00:00,  1.08s/it, loss=4.24]


epoch 1, loss:4.2401


Epoch 2: 100%|████████████████████████████████████████████████████████████| 843/843 [12:10<00:00,  1.15it/s, loss=3.18]


epoch 2, loss:3.1780


Epoch 3: 100%|████████████████████████████████████████████████████████████| 843/843 [12:44<00:00,  1.10it/s, loss=2.65]


...

epoch 61, loss:0.5162


Epoch 62: 100%|██████████████████████████████████████████████████████████| 843/843 [16:05<00:00,  1.15s/it, loss=0.516]


epoch 62, loss:0.5157


Epoch 63:  52%|██████████████████████████████▏                           | 438/843 [08:04<07:20,  1.09s/it, loss=0.512]

KeyboardInterrupt: 

In [15]:
feat_dim=train_set2[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set2[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
dropout_rate=0 #dropout in metric learning is much less recommended than in supervised learning
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate).to(device)
model.load_state_dict(torch.load("./model/human_mouse_lemur_60_epoch_model.pkl",weights_only=True))

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'human':3,'mouse':3,'lemur':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/human_mouse_lemur_60_epoch_loss_func.pkl",weights_only=True))

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=40
train(model,train_loaders2,val_loaders2,loss_function,optimizer,device,epochs)

Epoch 1: 100%|███████████████████████████████████████████████████████████| 843/843 [16:47<00:00,  1.19s/it, loss=0.528]


epoch 1, loss:0.5281


Epoch 2: 100%|███████████████████████████████████████████████████████████| 843/843 [15:01<00:00,  1.07s/it, loss=0.515]


epoch 2, loss:0.5151


Epoch 3: 100%|███████████████████████████████████████████████████████████| 843/843 [15:38<00:00,  1.11s/it, loss=0.514]


...

Epoch 40: 100%|██████████████████████████████████████████████████████████| 843/843 [15:05<00:00,  1.07s/it, loss=0.358]


epoch 40, loss:0.3585
At epoch 40 :


Species: human
accuracy:0.8237, f1_score:0.8255


Species: mouse
accuracy:0.9473, f1_score:0.9474


Species: lemur
accuracy:0.9800, f1_score:0.9787


### test

In [26]:
#test 95 epoch model
feat_dim=train_set2[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set2[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/human_mouse_lemur_95_epoch_model.pkl",weights_only=True))

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'human':3,'mouse':3,'lemur':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/human_mouse_lemur_95_epoch_loss_func.pkl",weights_only=True))

_=test(model,test_loaders2,loss_function,device)

Species: human
accuracy:0.8323, f1_score:0.8313


Species: mouse
accuracy:0.9460, f1_score:0.9462


Species: lemur
accuracy:0.9857, f1_score:0.9852


# For Get embeddings

In [18]:
class TiffDataset_with_metadata(Dataset):
    def __init__(self,original_dataset,image_cell_dict,label_celltype_dict):
        self.original_dataset=original_dataset
        self.image_cell_dict=image_cell_dict
        self.label_celltype_dict=label_celltype_dict
    def __len__(self):
        return len(self.original_dataset)
    def __getitem__(self,idx):
        original_return=self.original_dataset[idx]

        file_path,label=self.original_dataset.samples[idx]
        image_id=re.split(r'\\|/',file_path.strip())[-1]
        image_id=image_id.split('.')[0]
        species,cell_name=self.image_cell_dict[int(image_id)]

        cell_type=self.label_celltype_dict[species][int(label)]
        return original_return,{'cell_name':cell_name,'cell_type':cell_type}

In [19]:
def get_embeddings(model,sub_center_weights,test_loaders,num_sub_centers,label_celltype_dict,device):
    X_data=[]
    obs_df_data=[]
    model.eval()
    with torch.no_grad():
        for species,test_loader in test_loaders:
            for data,metadata in tqdm(test_loader,desc=f"Getting embeddings for {species}",leave=False):
                data={k:v.to(torch.float32).to(device) for k, v in data.items()}

                embeddings=model(data['feature_channels'],data['pos_channels']).cpu().numpy()
                for i in range(len(data['feature_channels'])):
                    X_data.append(embeddings[i])
                    obs_df_data.append([metadata['cell_name'][i],species,metadata['cell_type'][i]])

            s_weights=sub_center_weights[f'{species}.W'].T
            s_weights=nn.functional.normalize(s_weights,p=2,dim=-1).cpu().numpy()
            for i,subcenter_embedding in enumerate(s_weights):
                X_data.append(subcenter_embedding)

                label=math.floor(i/num_sub_centers[species])
                cell_type=f'{label_celltype_dict[species][label]}_subcenter'
                k_th=i%num_sub_centers[species]
                sub_center_name=f'{species}_{cell_type}_{k_th}'
                obs_df_data.append([sub_center_name,species,cell_type])
            print(f'Done get embeddings for {species}')
        torch.cuda.empty_cache()

    obs_df=pd.DataFrame(obs_df_data,columns=['cell_name','species','cell_type'])
    obs_df=obs_df.set_index('cell_name')
    adata=sc.AnnData(X=np.array(X_data),obs=obs_df)    
    return adata

# Get embeddings

## frog & zebrafish

### dataloader & prepare

In [15]:
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
all_species=['frog','zebrafish']

image_cell_dict={}
with open(f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_image_cell_correspondence.txt') as f:
    for line in f.readlines():
        items=line.strip().split('\t')
        image_id=int(items[0])
        species=items[1]
        cell_name=items[2]
        image_cell_dict[image_id]=[species,cell_name]

label_celltype_dict={}
for species in all_species:
    label_celltype_dict[species]={}
with open(f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_category_label_correspondence.txt') as f:
    for line in f.readlines():
        items=line.strip().split('\t')
        species=items[0]
        label=int(items[1])
        cell_type=items[2]
        label_celltype_dict[species][label]=cell_type

In [16]:
# frog_zebrafish 'with metadata' dataset building
# need to use the frog_zebrafish test set built before
all_species=['frog','zebrafish']

with_meta_test_set={}
for species in all_species:
    with_meta_test_set[species]=TiffDataset_with_metadata(test_set[species],image_cell_dict,label_celltype_dict)

In [17]:
# frog_zebrafish 'with metadata' dataloader building
all_species=['frog','zebrafish']

with_meta_test_loaders=[]
for species in all_species:
    _loader=DataLoader(dataset=with_meta_test_set[species],batch_size=32,shuffle=True,drop_last=False)
    with_meta_test_loaders.append((species,_loader))

### get embeddings

In [18]:
#use only test loaders
feat_dim=train_set[all_species[0]][0]['feature_channels'].shape[0]
pos_dim=train_set[all_species[0]][0]['pos_channels'].shape[0]
hid_dim=64
emb_dim=256
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/frog_zebrafish_95_epoch_model.pkl",weights_only=True))
sub_center_weights=torch.load("./model/frog_zebrafish_95_epoch_loss_func.pkl",weights_only=True)

num_sub_centers={'frog':5,'zebrafish':3}
embedding_adata=get_embeddings(model,sub_center_weights,with_meta_test_loaders,num_sub_centers,label_celltype_dict,device)
embedding_adata.write('./use_data/Save_for_Drawing/frog_zebrafish_95_epoch_embedding.h5ad')

Done get embeddings for frog


Done get embeddings for zebrafish


## human & mouse & lemur

### dataloader & prepare

In [21]:
mission_name='Human_Mouse_Lemur_2000hv_15000cell'
embedding_type='ESM1b'
all_species=['human','mouse','lemur']

image_cell_dict2={}
with open(f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_image_cell_correspondence.txt') as f:
    for line in f.readlines():
        items=line.strip().split('\t')
        image_id=int(items[0])
        species=items[1]
        cell_name=items[2]
        image_cell_dict2[image_id]=[species,cell_name]

label_celltype_dict2={}
for species in all_species:
    label_celltype_dict2[species]={}
with open(f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_category_label_correspondence.txt') as f:
    for line in f.readlines():
        items=line.strip().split('\t')
        species=items[0]
        label=int(items[1])
        cell_type=items[2]
        label_celltype_dict2[species][label]=cell_type

In [22]:
# need to use the human_mouse_lemur test set built before
all_species=['human','mouse','lemur']

with_meta_test_set2={}
for species in all_species:
    with_meta_test_set2[species]=TiffDataset_with_metadata(test_set2[species],image_cell_dict2,label_celltype_dict2)

In [23]:
# 'with metadata' dataloader building
all_species=['human','mouse','lemur']

with_meta_test_loaders2=[]
for species in all_species:
    _loader=DataLoader(dataset=with_meta_test_set2[species],batch_size=32,shuffle=True,drop_last=False)
    with_meta_test_loaders2.append((species,_loader))

### get embeddings

In [24]:
#use only test loaders
feat_dim=train_set2[all_species[0]][0]['feature_channels'].shape[0]
pos_dim=train_set2[all_species[0]][0]['pos_channels'].shape[0]
hid_dim=64
emb_dim=256
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/human_mouse_lemur_95_epoch_model.pkl",weights_only=True))
sub_center_weights=torch.load("./model/human_mouse_lemur_95_epoch_loss_func.pkl",weights_only=True)

num_sub_centers={'human':3,'mouse':3,'lemur':3}
embedding_adata=get_embeddings(model,sub_center_weights,with_meta_test_loaders2,num_sub_centers,label_celltype_dict2,device)
embedding_adata.write('./use_data/Save_for_Drawing/human_mouse_lemur_95_epoch_embedding.h5ad')

Done get embeddings for human


Done get embeddings for mouse


Done get embeddings for lemur
